In [ ]:
# ============================================================================
# CELL 1: Install Required Packages
# ============================================================================
# Run this cell first to install all necessary libraries
!pip install --upgrade langchain-chroma langchain-google-genai pypdf chromadb langchain-community langchain-core


# ============================================================================
# CELL 2: Import All Required Libraries
# ============================================================================

import os
import sys
from pathlib import Path  # For working with file paths
from google.colab import drive  # For mounting Google Drive
import warnings
warnings.filterwarnings('ignore')  # Suppress warning messages

# LangChain imports - core framework for RAG
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Splits documents into chunks
from langchain_community.document_loaders import (
    TextLoader,  # Loads text files (including .md)
    DirectoryLoader  # Loads all files from a directory
)
from langchain_chroma import Chroma  # Vector store (database for embeddings)
from langchain_google_genai import GoogleGenerativeAIEmbeddings  # Converts text to vectors
from langchain_google_genai import ChatGoogleGenerativeAI  # LLM for generating responses
from langchain_core.prompts import PromptTemplate  # Templates for structuring prompts
from langchain.chains import RetrievalQA  # RAG chain that retrieves and generates answers


# ============================================================================
# CELL 3: Mount Google Drive
# ============================================================================

drive.mount('/content/drive')
print("✓ Google Drive mounted successfully!")


# ============================================================================
# CELL 4: Set Up Paths and Configuration
# ============================================================================

# Define the base path for your project on Google Drive
BASE_PATH = Path("/content/drive/MyDrive/FirstAidRAG")

# Define paths for different folders
DOCS_PATH = BASE_PATH / "documents"  # Where your PDF/TXT files are stored
VECTOR_STORE_PATH = BASE_PATH / "vector_store"  # Where Chroma will save embeddings

# Create directories if they don't exist
DOCS_PATH.mkdir(parents=True, exist_ok=True)
VECTOR_STORE_PATH.mkdir(parents=True, exist_ok=True)

print(f"✓ Documents path: {DOCS_PATH}")
print(f"✓ Vector store path: {VECTOR_STORE_PATH}")

# Configuration settings
CHUNK_SIZE = 1000  # Size of each text chunk (in characters)
CHUNK_OVERLAP = 200  # Overlap between chunks (helps maintain context)
MODEL_NAME = "models/embedding-001"  # Google's free embedding model
EMBEDDINGS_CACHE_PATH = BASE_PATH / "embeddings_cache"  # Cache embeddings to avoid API calls

# Create embeddings cache directory
EMBEDDINGS_CACHE_PATH.mkdir(parents=True, exist_ok=True)

print("\n📋 Configuration:")
print(f"  - Chunk size: {CHUNK_SIZE} characters")
print(f"  - Chunk overlap: {CHUNK_OVERLAP} characters")
print(f"  - Embedding model: {MODEL_NAME}")
print(f"  - Embeddings cache: {EMBEDDINGS_CACHE_PATH}")


# ============================================================================
# CELL 5: Load Documents from Folder
# ============================================================================

def load_documents(doc_path):
    """
    Load all markdown documents from the specified directory using DirectoryLoader.
    This is efficient for loading multiple .md files at once.
    
    Args:
        doc_path: Path to the documents folder
    
    Returns:
        List of Document objects with content and metadata
    """
    
    print("\n📂 Loading markdown documents...")
    
    # Use DirectoryLoader to load all .md files from the directory
    # glob pattern "**/*.md" means: search for all .md files recursively
    loader = DirectoryLoader(
        path=str(doc_path),
        glob="**/*.md",  # Load only .md files
        loader_cls=TextLoader,  # Use TextLoader to read each file
        loader_kwargs={"encoding": "utf-8"}  # Use UTF-8 encoding for reading
    )
    
    # Load all documents
    documents = loader.load()
    
    print(f"✓ Loaded {len(documents)} markdown files")
    return documents


# Load all documents
all_documents = load_documents(DOCS_PATH)

if len(all_documents) == 0:
    print("⚠️  WARNING: No documents found! Please upload PDFs or TXT files to the documents folder.")


# ============================================================================
# CELL 6: Split Documents into Chunks
# ============================================================================

def split_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """
    Split documents into smaller chunks for better retrieval.
    Chunks have overlap to maintain context across boundaries.
    
    Args:
        documents: List of Document objects
        chunk_size: Size of each chunk in characters
        chunk_overlap: Number of overlapping characters between chunks
    
    Returns:
        List of Document chunks
    """
    
    print("\n✂️  Splitting documents into chunks...")
    
    # Initialize the text splitter
    # RecursiveCharacterTextSplitter tries to split at sentence/paragraph boundaries first
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,  # Each chunk will be ~1000 characters
        chunk_overlap=chunk_overlap,  # 200 character overlap between chunks
        separators=["\n\n", "\n", ". ", " ", ""]  # Try to split at these boundaries in order
    )
    
    # Split all documents
    chunks = text_splitter.split_documents(documents)
    
    print(f"✓ Created {len(chunks)} chunks")
    print(f"  Average chunk size: {sum(len(chunk.page_content) for chunk in chunks) // len(chunks)} characters")
    
    return chunks


# Split the documents
document_chunks = split_documents(all_documents)


# ============================================================================
# CELL 7: Set Up Google API Key and Initialize Embedding Model
# ============================================================================

# Get your Google API key
# Visit: https://makersuite.google.com/app/apikey to get a free API key
google_api_key = input("Enter your Google API Key (from https://makersuite.google.com/app/apikey): ")

# Set the API key as an environment variable
os.environ["GOOGLE_API_KEY"] = google_api_key

print("✓ API key set!")

# Initialize the embedding model
print("\n🔧 Initializing embedding model...")
embeddings = GoogleGenerativeAIEmbeddings(
    model=MODEL_NAME,  # Use Google's free embedding model
    google_api_key=google_api_key
)
print("✓ Embedding model initialized!")


# ============================================================================
# CELL 7B: Cache Embeddings Locally (Avoid Repeated API Calls)
# ============================================================================

import json
from datetime import datetime

def save_embedding_cache(cache_path, doc_count):
    """
    Save metadata about current embeddings to track what's been cached.
    This helps avoid re-embedding the same documents.
    """
    cache_info = {
        "timestamp": datetime.now().isoformat(),
        "document_count": doc_count,
        "model": MODEL_NAME
    }
    with open(cache_path / "cache_info.json", "w") as f:
        json.dump(cache_info, f, indent=2)
    print("✓ Embedding cache info saved!")

def load_embedding_cache(cache_path):
    """
    Load cached embedding metadata to check if re-embedding is needed.
    """
    cache_file = cache_path / "cache_info.json"
    if cache_file.exists():
        with open(cache_file, "r") as f:
            return json.load(f)
    return None

# Check if embeddings were already cached
cached_info = load_embedding_cache(EMBEDDINGS_CACHE_PATH)
current_doc_count = len(document_chunks)

if cached_info:
    print(f"\n💾 Found cached embeddings!")
    print(f"  - Cached documents: {cached_info['document_count']}")
    print(f"  - Current documents: {current_doc_count}")
    if cached_info['document_count'] == current_doc_count:
        print("  - Status: ✅ All documents already embedded (no new API calls)")
    else:
        print("  - Status: ⚠️  New documents detected, will create new embeddings")
else:
    print(f"\n⚠️  No cached embeddings found. First run will use API calls.")
    print(f"  - Documents to embed: {current_doc_count}")
    print(f"  - Estimated API calls: {current_doc_count} (will be cached after)")


# ============================================================================
# CELL 8: Create or Load Vector Store
# ============================================================================

def create_or_load_vector_store(chunks, embeddings, persist_path):
    """
    Create a new vector store or load an existing one from disk.
    
    Args:
        chunks: Document chunks to embed (used only for creation)
        embeddings: Embedding model
        persist_path: Path where vector store is/will be saved
    
    Returns:
        Chroma vector store object
    """
    
    print("\n🗄️  Vector Store Setup...")
    
    # Check if vector store already exists
    if (persist_path / "chroma.sqlite3").exists():
        # Load existing vector store (much faster!)
        print("Found existing vector store, loading...")
        vector_store = Chroma(
            persist_directory=str(persist_path),
            embedding_function=embeddings
        )
        print(f"✓ Loaded existing vector store with {vector_store._collection.count()} vectors")
        
    else:
        # Create new vector store from chunks
        print("Creating new vector store from documents...")
        print("This may take a few minutes on first run...")
        
        vector_store = Chroma.from_documents(
            documents=chunks,  # Your document chunks
            embedding=embeddings,  # Embedding model
            persist_directory=str(persist_path),  # Save location
            collection_name="first_aid_docs"  # Name of the collection
        )
        print(f"✓ Created new vector store with {len(chunks)} documents")
        print(f"✓ Vector store saved to: {persist_path}")
    
    return vector_store


# Create or load the vector store
vector_store = create_or_load_vector_store(
    document_chunks, 
    embeddings, 
    VECTOR_STORE_PATH
)

# Save embedding cache info
save_embedding_cache(EMBEDDINGS_CACHE_PATH, len(document_chunks))


# ============================================================================
# CELL 9: Set Up the Language Model (LLM)
# ============================================================================

print("\n🤖 Initializing Language Model...")

# Initialize Google's Generative AI model (Gemini)
llm = ChatGoogleGenerativeAI(
    model="gemini-pro",  # Use Gemini Pro (free tier available)
    google_api_key=google_api_key,
    temperature=0.7  # 0.7 = balanced between creative and factual
)

print("✓ Language Model initialized!")
print("  Model: Gemini Pro")
print("  Temperature: 0.7 (balanced)")


# ============================================================================
# CELL 10: Create Custom Prompt Template
# ============================================================================

# Define a custom prompt that tells the LLM how to answer questions
CUSTOM_PROMPT = """You are a helpful First Aid assistant. Answer questions based ONLY on the provided context.

Context:
{context}

Question: {question}

Instructions:
1. Provide clear, concise answers
2. Use the provided context/documents to answer
3. If the answer is not in the context, say "I don't have information about this in my knowledge base"
4. For emergency situations, always recommend calling emergency services
5. Be empathetic and professional

Answer:"""

# Create a PromptTemplate object
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=CUSTOM_PROMPT
)

print("\n📝 Custom prompt template created!")


# ============================================================================
# CELL 11: Build the RAG Chain
# ============================================================================

print("\n🔗 Building RAG Chain...")

# Create a retriever from the vector store
# This will search for relevant documents based on similarity
retriever = vector_store.as_retriever(
    search_type="similarity",  # Use similarity search
    search_kwargs={"k": 3}  # Return top 3 most relevant documents
)

# Create the RAG chain using RetrievalQA
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,  # The language model
    chain_type="stuff",  # "stuff" means combine all docs into one prompt
    retriever=retriever,  # The document retriever
    return_source_documents=True,  # Return which documents were used
    chain_type_kwargs={"prompt": prompt_template}
)

print("✓ RAG Chain created successfully!")
print("  - Retriever: Similarity search (top 3 documents)")
print("  - Chain type: Stuff (combines all retrieved docs)")
print("  - Returns source documents: Yes")


# ============================================================================
# CELL 12: Create Chatbot Function
# ============================================================================

def ask_first_aid_question(question):
    """
    Ask the First Aid RAG chatbot a question and get an answer.
    
    Args:
        question: The user's question
    
    Returns:
        Dictionary with answer and source documents
    """
    
    print("\n" + "="*70)
    print(f"❓ Question: {question}")
    print("="*70)
    
    try:
        # Run the RAG chain
        response = rag_chain.invoke({"query": question})
        
        # Extract answer and sources
        answer = response["result"]
        source_docs = response["source_documents"]
        
        # Display the answer
        print(f"\n✅ Answer:\n{answer}")
        
        # Display sources
        print(f"\n📚 Sources Used ({len(source_docs)} documents):")
        for i, doc in enumerate(source_docs, 1):
            print(f"\n  Source {i}:")
            print(f"    File: {doc.metadata.get('source', 'Unknown')}")
            print(f"    Preview: {doc.page_content[:200]}...")
        
        print("\n" + "="*70 + "\n")
        
        return {
            "question": question,
            "answer": answer,
            "sources": source_docs
        }
        
    except Exception as e:
        print(f"\n❌ Error: {str(e)}")
        return None


# ============================================================================
# CELL 13: Test the Chatbot with Sample Questions
# ============================================================================

# Test with sample questions about first aid
test_questions = [
    "What should I do if someone is choking?",
    "How do I perform CPR?",
    "What is the treatment for a minor burn?",
    "How should I treat a bleeding wound?",
]

print("\n🧪 Testing the First Aid RAG Chatbot...\n")

# Ask the first question
result = ask_first_aid_question(test_questions[0])

# Uncomment below to test more questions
# for question in test_questions[1:]:
#     result = ask_first_aid_question(question)


# ============================================================================
# CELL 14: Interactive Chat Loop (Optional)
# ============================================================================

print("\n" + "="*70)
print("🎯 INTERACTIVE FIRST AID CHATBOT")
print("="*70)
print("Type your first aid questions below.")
print("Type 'quit' or 'exit' to stop.\n")

while True:
    # Get user input
    user_question = input("You: ").strip()
    
    # Check for exit commands
    if user_question.lower() in ['quit', 'exit', 'q']:
        print("\nGoodbye! Stay safe and always call emergency services in critical situations.")
        break
    
    # Check if question is empty
    if not user_question:
        print("Please enter a question.\n")
        continue
    
    # Ask the question
    ask_first_aid_question(user_question)